In [54]:
import numpy as np
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [23]:
df = pd.read_csv('/content/SMSSpamCollection', sep='\t', header=None, names=['label', 'text'])

In [24]:
df.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [33]:
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [35]:
stop_words = set(stopwords.words('english'))

In [36]:
def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()

    # 2. Remove punctuation, digits, special symbols
    text = re.sub(r'[^a-z\s]', '', text)

    # 3. Tokenize
    tokens = word_tokenize(text)

    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    return tokens

In [39]:
df['tokens'] = df['text'].apply(preprocess_text)
df[['label', 'tokens']].head(10)

,label,tokens
0,ham,"[go, jurong, point, crazy, available, bugis, n..."
1,ham,"[ok, lar, joking, wif, u, oni]"
2,spam,"[free, entry, wkly, comp, win, fa, cup, final,..."
3,ham,"[u, dun, say, early, hor, u, c, already, say]"
4,ham,"[nah, dont, think, goes, usf, lives, around, t..."
5,spam,"[freemsg, hey, darling, weeks, word, back, id,..."
6,ham,"[even, brother, like, speak, treat, like, aids..."
7,ham,"[per, request, melle, melle, oru, minnaminungi..."
8,spam,"[winner, valued, network, customer, selected, ..."
9,spam,"[mobile, months, u, r, entitled, update, lates..."


In [41]:
df['clean_text'] = df['tokens'].apply(lambda x: ' '.join(x))

In [43]:
tfidf_vectorizer = TfidfVectorizer()

# Fit on training data and transform
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

print("TF-IDF Matrix Shape:", X_tfidf.shape)


TF-IDF Matrix Shape: (5572, 8476)


In [45]:
from sklearn.model_selection import train_test_split

# Features and labels
X = X_tfidf
y = df['label']

# Train/test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [53]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Predictions
y_pred_nb = nb_model.predict(X_test)

# Evaluation
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, y_pred_nb):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9623

Classification Report:
               precision    recall  f1-score   support

         ham       0.96      1.00      0.98       966
        spam       1.00      0.72      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.86      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [51]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.9587

Classification Report:
               precision    recall  f1-score   support

         ham       0.95      1.00      0.98       966
        spam       1.00      0.69      0.82       149

    accuracy                           0.96      1115
   macro avg       0.98      0.85      0.90      1115
weighted avg       0.96      0.96      0.96      1115



In [55]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [56]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [64]:
nb_model = MultinomialNB()

nb_scores = cross_val_score(nb_model, X, y, cv=kfold, scoring='accuracy')

print("Naive Bayes Cross-Validation Results:")
print("Scores for each fold:", np.round(nb_scores, 4))
print(f"Mean Accuracy: {nb_scores.mean():.4f}")
print(f"Standard Deviation: {nb_scores.std():.4f}")

Naive Bayes Cross-Validation Results:
Scores for each fold: [0.9695 0.9614 0.9641 0.9578 0.9659]
Mean Accuracy: 0.9637
Standard Deviation: 0.0040


In [63]:
lr_model = LogisticRegression(max_iter=1000)

lr_scores = cross_val_score(lr_model, X, y, cv=kfold, scoring='accuracy')

print("\nLogistic Regression Cross-Validation Results:")
print("Scores for each fold:", np.round(lr_scores, 4))
print(f"Mean Accuracy: {lr_scores.mean():.4f}")
print(f"Standard Deviation: {lr_scores.std():.4f}")


Logistic Regression Cross-Validation Results:
Scores for each fold: [0.9561 0.9605 0.9506 0.9461 0.956 ]
Mean Accuracy: 0.9539
Standard Deviation: 0.0050
